In [1]:
# notebooks/EDA.ipynb

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set(font_scale=1.1)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


In [ ]:
data_path = "../data/dataset_2.csv"
df = pd.read_csv(data_path)

df.head()
df.info()
df.describe(include="all")


In [ ]:
# Strip whitespace from string columns
str_cols = df.select_dtypes(include="object").columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

# Standardize boolean column
df["Has_Pool"] = df["Has_Pool"].map({"Yes": 1, "No": 0})

# Check missing values
df.isna().sum()


In [ ]:
# Numeric columns: Area_SqFt, Rooms, Build_Year, Price
num_cols = ["Area_SqFt", "Rooms", "Build_Year", "Price"]
cat_cols = ["Location", "Street_Type", "Furnishing", "Property_Type"]

# Impute numeric with median
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Impute categorical with mode
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

df.isna().sum()


In [ ]:
# Age of property
current_year = 2024
df["Property_Age"] = current_year - df["Build_Year"]

# Price per SqFt
df["Price_per_SqFt"] = df["Price"] / df["Area_SqFt"]

df[["Area_SqFt", "Price", "Property_Age", "Price_per_SqFt"]].head()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

sns.histplot(df["Price"], kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Price Distribution")

sns.histplot(df["Area_SqFt"], kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Area Distribution")

sns.countplot(x="Rooms", data=df, ax=axes[1, 0])
axes[1, 0].set_title("Rooms Count")

sns.countplot(x="Location", data=df, ax=axes[1, 1])
axes[1, 1].set_title("Location Count")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
corr_cols = ["Area_SqFt", "Rooms", "Build_Year", "Has_Pool", "Property_Age", "Price_per_SqFt", "Price"]
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(x="Area_SqFt", y="Price", data=df)
plt.title("Price vs Area")
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(x="Rooms", y="Price", data=df)
plt.title("Price vs Rooms")
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x="Location", y="Price", data=df)
plt.title("Price vs Location")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x="Property_Type", y="Price", data=df)
plt.title("Price vs Property Type")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x="Furnishing", y="Price", data=df)
plt.title("Price vs Furnishing")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(x="Has_Pool", y="Price", data=df)
plt.title("Price vs Pool (0=No, 1=Yes)")
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x="Street_Type", y="Price", data=df)
plt.title("Price vs Street Type")
plt.xticks(rotation=45)
plt.show()


In [ ]:
year_price = df.groupby("Build_Year")["Price"].mean().reset_index()

plt.figure(figsize=(10, 4))
sns.lineplot(x="Build_Year", y="Price", data=year_price, marker="o")
plt.title("Average Price by Build Year")
plt.show()


In [11]:
clean_path = "../data/cleaned_dataset.csv"
df.to_csv(clean_path, index=False)
clean_path


'../data/cleaned_dataset.csv'

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

df_model = df.copy()

# Encode categorical columns
cat_cols = ["Location", "Street_Type", "Furnishing", "Property_Type"]
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le

# Features & target
X = df_model.drop("Price", axis=1)
y = df_model["Price"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

# Save model & encoders
import pickle
pickle.dump(model, open("../app/price_model.pkl", "wb"))
pickle.dump(encoders, open("../app/encoders.pkl", "wb"))


MAE: 26699.200862448524
R2 Score: 0.8936030934068857
